In [ ]:
#install all the necessary packages

!pip install -q bitsandbytes transformers peft accelerate
!pip install -q datasets trl ninja packaging


In [ ]:
import torch
import os
import sys
import json
import IPython
from datetime import datetime
from datasets import load_dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoTokenizer,
    TrainingArguments,
)
from trl import SFTTrainer

In [ ]:
from datasets import load_dataset

## Alpaca dataset
dataset = load_dataset("yahma/alpaca-cleaned")
print(dataset)

# inspect one example
example = dataset["train"][8]

print("Instruction:", example["instruction"])
print("Input:", example["input"])
print("Output:", example["output"])


In [ ]:
model_name = "Qwen/Qwen2-0.5B"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

print(model.num_parameters())
print(model)


In [ ]:
# === evaluate LLMs by generating text for prompts ===
prompt = """### Instruction:
Write a Python function that checks whether a number is prime.

### Response:
"""
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from datasets import load_dataset, DatasetDict

### merge all three fields into one 'text' field for SFTTrain
def merge_all(example):
    instruction = example["instruction"].strip()
    input_text = example["input"].strip()
    output_text = example["output"].strip()

    if input_text:
        example["text"] = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{output_text}"""
    else:
        example["text"] = f"""### Instruction:
{instruction}

### Response:
{output_text}"""

    return example

ds = load_dataset("yahma/alpaca-cleaned")["train"]

# shuffle and take exactly 7000 examples total
ds = ds.shuffle(seed=42).select(range(7000))

# create the text field
ds = ds.map(merge_all)


train_dataset = ds.select(range(5000))
test_dataset = ds.select(range(5000, 7000))

print(ds)
print(train_dataset[0]["text"])
print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

In [ ]:
from trl import SFTTrainer, SFTConfig

compute_dtype = getattr(torch, "float16")
print(compute_dtype)
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
)

In [ ]:
#Quantization as defined https://huggingface.co/docs/optimum/concept_guides/quantization will help us reduce the size of the model for it to fit on a single GPU
#Quantization configuration

from trl import SFTTrainer, SFTConfig

compute_dtype = getattr(torch, "float16")
print(compute_dtype)
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
)

#Load the model and quantize it
model = AutoModelForCausalLM.from_pretrained(
          model_name,
          quantization_config=bnb_config,
          device_map={"": 0}, #device_map="auto" will cause a problem in the training

)

print(model.num_parameters())
print(model)

In [ ]:
### config for LoRa
peft_config = LoraConfig(
        lora_alpha=16,
        lora_dropout=0.05,
        r=16,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules= ['k_proj', 'q_proj', 'v_proj', 'o_proj', "gate_proj", "down_proj", "up_proj", "lm_head",]
)

#Cast some modules of the model to fp32

model = prepare_model_for_kbit_training(model)
#Configure the pad token in the model
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False # Gradient checkpointing is used by default but not compatible with caching


In [ ]:
training_arguments = SFTConfig(
        output_dir="./results",
        eval_strategy="epoch",
        optim="paged_adamw_8bit",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        save_steps=500,
        logging_steps=20,
        learning_rate=4e-4,
        num_train_epochs=1,
        warmup_steps=100,
        lr_scheduler_type="constant",
        report_to="none",
        max_length=512,
        packing = True,
        dataset_text_field="text",
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    args=training_arguments,
)


trainer.train()
trainer.save_model("./final_lora_adapter")

In [ ]:
# reload the base model and the LoRA adaptor

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, "./final_lora_adapter")
model.eval()

In [ ]:
# === evaluate the fine-tuned LLMs for prompts ===
prompt = """### Instruction:
Write a Python function that checks whether a number is prime.

### Response:
"""
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
import math

train_metrics = trainer.evaluate(eval_dataset=trainer.train_dataset, metric_key_prefix="train")
test_metrics = trainer.evaluate(eval_dataset=trainer.eval_dataset, metric_key_prefix="test")

train_loss = train_metrics["train_loss"]
test_loss = test_metrics["test_loss"]

train_ppl = math.exp(train_loss)
test_ppl = math.exp(test_loss)

print("Train loss:", train_loss)
print("Train perplexity:", train_ppl)
print("Test loss:", test_loss)
print("Test perplexity:", test_ppl)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

size_results = []
rank_results = []

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
)

r_value = 16

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=r_value,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'q_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'down_proj', 'up_proj', 'lm_head']
)

model = prepare_model_for_kbit_training(model)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

training_arguments = SFTConfig(
    output_dir="./results_size_500",
    eval_strategy="epoch",
    optim="paged_adamw_8bit",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    log_level="error",
    save_steps=500,
    logging_steps=20,
    learning_rate=4e-4,
    num_train_epochs=1,
    warmup_steps=100,
    lr_scheduler_type="constant",
    report_to="none",
    max_length=512,
    packing=True,
    dataset_text_field="text",
)

train_dataset_n = train_dataset.select(range(500))

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_n,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    args=training_arguments,
)

trainer.train()

train_metrics = trainer.evaluate(eval_dataset=trainer.train_dataset, metric_key_prefix="train")
test_metrics = trainer.evaluate(eval_dataset=trainer.eval_dataset, metric_key_prefix="test")

train_loss = train_metrics["train_loss"]
test_loss = test_metrics["test_loss"]

train_ppl = math.exp(train_loss)
test_ppl = math.exp(test_loss)

print("Train perplexity (500):", train_ppl)
print("Test perplexity (500):", test_ppl)

size_results.append({
    "train_size": 500,
    "train_ppl": train_ppl,
    "test_ppl": test_ppl
})

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
)

r_value = 16

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=r_value,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'q_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'down_proj', 'up_proj', 'lm_head']
)

model = prepare_model_for_kbit_training(model)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

training_arguments = SFTConfig(
    output_dir="./results_size_1000",
    eval_strategy="epoch",
    optim="paged_adamw_8bit",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    log_level="error",
    save_steps=500,
    logging_steps=20,
    learning_rate=4e-4,
    num_train_epochs=1,
    warmup_steps=100,
    lr_scheduler_type="constant",
    report_to="none",
    max_length=512,
    packing=True,
    dataset_text_field="text",
)

train_dataset_n = train_dataset.select(range(1000))

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_n,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    args=training_arguments,
)

trainer.train()

train_metrics = trainer.evaluate(eval_dataset=trainer.train_dataset, metric_key_prefix="train")
test_metrics = trainer.evaluate(eval_dataset=trainer.eval_dataset, metric_key_prefix="test")

train_loss = train_metrics["train_loss"]
test_loss = test_metrics["test_loss"]

train_ppl = math.exp(train_loss)
test_ppl = math.exp(test_loss)

print("Train perplexity (1000):", train_ppl)
print("Test perplexity (1000):", test_ppl)

size_results.append({
    "train_size": 1000,
    "train_ppl": train_ppl,
    "test_ppl": test_ppl
})

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
)

r_value = 16

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=r_value,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'q_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'down_proj', 'up_proj', 'lm_head']
)

model = prepare_model_for_kbit_training(model)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

training_arguments = SFTConfig(
    output_dir="./results_size_2000",
    eval_strategy="epoch",
    optim="paged_adamw_8bit",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    log_level="error",
    save_steps=500,
    logging_steps=20,
    learning_rate=4e-4,
    num_train_epochs=1,
    warmup_steps=100,
    lr_scheduler_type="constant",
    report_to="none",
    max_length=512,
    packing=True,
    dataset_text_field="text",
)

train_dataset_n = train_dataset.select(range(2000))

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_n,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    args=training_arguments,
)

trainer.train()

train_metrics = trainer.evaluate(eval_dataset=trainer.train_dataset, metric_key_prefix="train")
test_metrics = trainer.evaluate(eval_dataset=trainer.eval_dataset, metric_key_prefix="test")

train_loss = train_metrics["train_loss"]
test_loss = test_metrics["test_loss"]

train_ppl = math.exp(train_loss)
test_ppl = math.exp(test_loss)

print("Train perplexity (2000):", train_ppl)
print("Test perplexity (2000):", test_ppl)

size_results.append({
    "train_size": 2000,
    "train_ppl": train_ppl,
    "test_ppl": test_ppl
})
torch.cuda.empty_cache()

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
)

r_value = 16

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=r_value,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'q_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'down_proj', 'up_proj', 'lm_head']
)

model = prepare_model_for_kbit_training(model)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

training_arguments = SFTConfig(
    output_dir="./results_size_5000",
    eval_strategy="epoch",
    optim="paged_adamw_8bit",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    log_level="error",
    save_steps=500,
    logging_steps=20,
    learning_rate=4e-4,
    num_train_epochs=1,
    warmup_steps=100,
    lr_scheduler_type="constant",
    report_to="none",
    max_length=512,
    packing=True,
    dataset_text_field="text",
)

train_dataset_n = train_dataset.select(range(5000))

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_n,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    args=training_arguments,
)

trainer.train()

train_metrics = trainer.evaluate(eval_dataset=trainer.train_dataset, metric_key_prefix="train")
test_metrics = trainer.evaluate(eval_dataset=trainer.eval_dataset, metric_key_prefix="test")

train_loss = train_metrics["train_loss"]
test_loss = test_metrics["test_loss"]

train_ppl = math.exp(train_loss)
test_ppl = math.exp(test_loss)

print("Train perplexity (5000):", train_ppl)
print("Test perplexity (5000):", test_ppl)

size_results.append({
    "train_size": 5000,
    "train_ppl": train_ppl,
    "test_ppl": test_ppl
})
torch.cuda.empty_cache()

In [ ]:
df_sizes = pd.DataFrame(size_results)
print(df_sizes)

plt.figure(figsize=(8, 5))
plt.plot(df_sizes["train_size"], df_sizes["train_ppl"], marker="o", label="Train perplexity")
plt.plot(df_sizes["train_size"], df_sizes["test_ppl"], marker="o", label="Test perplexity")
plt.xlabel("Number of training samples")
plt.ylabel("Perplexity")
plt.title("Perplexity vs Training Set Size")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
torch.cuda.empty_cache()
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
)

r_value = 4

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=r_value,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'q_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'down_proj', 'up_proj', 'lm_head']
)

model = prepare_model_for_kbit_training(model)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

training_arguments = SFTConfig(
    output_dir="./results_rank_4",
    eval_strategy="epoch",
    optim="paged_adamw_8bit",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    log_level="error",
    save_steps=500,
    logging_steps=20,
    learning_rate=4e-4,
    num_train_epochs=1,
    warmup_steps=100,
    lr_scheduler_type="constant",
    report_to="none",
    max_length=512,
    packing=True,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    args=training_arguments,
)

trainer.train()

train_metrics = trainer.evaluate(eval_dataset=trainer.train_dataset, metric_key_prefix="train")
test_metrics = trainer.evaluate(eval_dataset=trainer.eval_dataset, metric_key_prefix="test")

train_loss = train_metrics["train_loss"]
test_loss = test_metrics["test_loss"]

train_ppl = math.exp(train_loss)
test_ppl = math.exp(test_loss)

trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
peak_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

print("Train perplexity (r=4):", train_ppl)
print("Test perplexity (r=4):", test_ppl)
print("Trainable params (r=4):", trainable_params)
print("Peak GPU memory GB (r=4):", peak_memory_gb)

rank_results.append({
    "rank": 4,
    "train_ppl": train_ppl,
    "test_ppl": test_ppl,
    "trainable_params": trainable_params,
    "peak_memory_gb": peak_memory_gb
})
torch.cuda.empty_cache()

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
)

r_value = 8

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=r_value,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'q_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'down_proj', 'up_proj', 'lm_head']
)

model = prepare_model_for_kbit_training(model)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

training_arguments = SFTConfig(
    output_dir="./results_rank_8",
    eval_strategy="epoch",
    optim="paged_adamw_8bit",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    log_level="error",
    save_steps=500,
    logging_steps=20,
    learning_rate=4e-4,
    num_train_epochs=1,
    warmup_steps=100,
    lr_scheduler_type="constant",
    report_to="none",
    max_length=512,
    packing=True,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    args=training_arguments,
)

trainer.train()

train_metrics = trainer.evaluate(eval_dataset=trainer.train_dataset, metric_key_prefix="train")
test_metrics = trainer.evaluate(eval_dataset=trainer.eval_dataset, metric_key_prefix="test")

train_loss = train_metrics["train_loss"]
test_loss = test_metrics["test_loss"]

train_ppl = math.exp(train_loss)
test_ppl = math.exp(test_loss)

trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
peak_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

print("Train perplexity (r=8):", train_ppl)
print("Test perplexity (r=8):", test_ppl)
print("Trainable params (r=8):", trainable_params)
print("Peak GPU memory GB (r=8):", peak_memory_gb)

rank_results.append({
    "rank": 8,
    "train_ppl": train_ppl,
    "test_ppl": test_ppl,
    "trainable_params": trainable_params,
    "peak_memory_gb": peak_memory_gb
})
torch.cuda.empty_cache()

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
)

r_value = 16

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=r_value,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'q_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'down_proj', 'up_proj', 'lm_head']
)

model = prepare_model_for_kbit_training(model)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

training_arguments = SFTConfig(
    output_dir="./results_rank_16",
    eval_strategy="epoch",
    optim="paged_adamw_8bit",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    log_level="error",
    save_steps=500,
    logging_steps=20,
    learning_rate=4e-4,
    num_train_epochs=1,
    warmup_steps=100,
    lr_scheduler_type="constant",
    report_to="none",
    max_length=512,
    packing=True,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    args=training_arguments,
)

trainer.train()

train_metrics = trainer.evaluate(eval_dataset=trainer.train_dataset, metric_key_prefix="train")
test_metrics = trainer.evaluate(eval_dataset=trainer.eval_dataset, metric_key_prefix="test")

train_loss = train_metrics["train_loss"]
test_loss = test_metrics["test_loss"]

train_ppl = math.exp(train_loss)
test_ppl = math.exp(test_loss)

trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
peak_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

print("Train perplexity (r=16):", train_ppl)
print("Test perplexity (r=16):", test_ppl)
print("Trainable params (r=16):", trainable_params)
print("Peak GPU memory GB (r=16):", peak_memory_gb)

rank_results.append({
    "rank": 16,
    "train_ppl": train_ppl,
    "test_ppl": test_ppl,
    "trainable_params": trainable_params,
    "peak_memory_gb": peak_memory_gb
})

In [ ]:
df_ranks = pd.DataFrame(rank_results)
print(df_ranks)

plt.figure(figsize=(8, 5))
plt.plot(df_ranks["rank"], df_ranks["train_ppl"], marker="o", label="Train perplexity")
plt.plot(df_ranks["rank"], df_ranks["test_ppl"], marker="o", label="Test perplexity")
plt.xlabel("LoRA rank")
plt.ylabel("Perplexity")
plt.title("Perplexity vs LoRA Rank")
plt.legend()
plt.grid(True)
plt.show()